In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from gostplot import GostPlot


In [ ]:
# Исходные данные: N — отсчёты за время t (мин), l — толщина поглотителя (см).
# ВАЖНО: толщины заданы явными массивами, а не range(), т.к. часть точек
# выброшена (сбои установки) и равномерный шаг больше не гарантирован.

# --- свинец ---------------------------------------------------------------
lead   = np.array([218169, 120967, 137538, 123208, 120725])
lead_t = np.array([1, 1, 2, 3, 5])
lead_l = np.array([0.0, 0.5, 1.0, 1.5, 2.0])

# --- алюминий -------------------------------------------------------------
aluminium   = np.array([380478, 244971, 165074, 218222, 144653, 146486, 98846, 90551])
aluminium_t = np.array([1, 1, 1, 2, 2, 3, 3, 4])
aluminium_l = np.array([0., 2., 4., 6., 8., 10., 12., 14.])

# --- сталь ----------------------------------------------------------------
# последняя точка (l = 7 см, 210545 отсчётов за 15 мин) даёт скорость счёта
# ВЫШЕ предыдущей => промах, в аппроксимацию не берём (см. steel_ok).
steel   = np.array([379401, 211783, 119095, 136349, 118095, 139596, 138560, 210545])
steel_t = np.array([1, 1, 1, 2, 3, 6, 10, 15])
steel_l = np.array([0., 1., 2., 3., 4., 5., 6., 7.])
steel_ok = np.array([True]*7 + [False])

# --- фон ------------------------------------------------------------------
bg_counts = 2859 + 3085       # суммарные отсчёты фона
bg_time   = 6.0               # суммарное время набора фона, мин
zero_gamma_level   = bg_counts / bg_time                # N0, 1/мин
sigma_zero_gamma   = np.sqrt(bg_counts) / bg_time       # sigma(N0), 1/мин

# Плотности, г/см^3
density = {'Al': 2.70, 'Fe': 7.87, 'Cu': 8.96, 'Pb': 11.34}

# Табличные массовые коэффициенты ослабления при E = 662 кэВ (Cs-137), см^2/г.
# СВЕРИТЬ с таблицей в методичке перед сдачей!
mu_rho_tab = {'Al': 0.0744, 'Fe': 0.0731, 'Pb': 0.111}

# Скорости счёта, 1/мин
lead_s      = lead / lead_t
aluminium_s = aluminium / aluminium_t
steel_s     = steel / steel_t


In [ ]:
def net_and_sigma(counts, t):
    """Скорость счёта за вычетом фона и её погрешность.

    n  = N/t,            sigma(n)  = sqrt(N)/t
    n0 = N0/t0,          sigma(n0) = sqrt(N0)/t0
    """
    n     = counts / t
    s_n   = np.sqrt(counts) / t
    net   = n - zero_gamma_level
    s_net = np.hypot(s_n, sigma_zero_gamma)
    return net, s_net


def ln_and_sigma(counts, t):
    """ln(n - n0) и его погрешность: sigma(ln x) = sigma(x)/x."""
    net, s_net = net_and_sigma(counts, t)
    return np.log(net), s_net / net


lead_ln,      sigma_lead_ln      = ln_and_sigma(lead, lead_t)
aluminium_ln, sigma_aluminium_ln = ln_and_sigma(aluminium, aluminium_t)
steel_ln,     sigma_steel_ln     = ln_and_sigma(steel, steel_t)


In [ ]:
# Погрешность толщины: штангенциркуль, цена деления 0.1 мм -> половина деления
dl = 0.005   # см

# ВНИМАНИЕ: .errors(...) вызывается ПОСЛЕ .fit(1) намеренно.
# Погрешность счёта ~0.3 %, но точки ложатся на прямую заметно хуже
# (chi2/ndf >> 1: накопление рассеянного излучения + разброс толщин).
# Взвешенный фит выдал бы sigma(mu) в ~10 раз меньше реального разброса.
# Так усы рисуются, а погрешность наклона считается по остаткам — честно.

plot_pb = (GostPlot(lead_l, lead_ln)
           .xlabel(r'$l$', 'см')
           .ylabel(r'$\ln\left[(N-N_0)/\mathrm{мин}^{-1}\right]$')
           .title('Ослабление $\\gamma$-излучения: свинец')
           .fit(1)
           .errors(dy=sigma_lead_ln, dx=dl))

plot_al = (GostPlot(aluminium_l, aluminium_ln)
           .xlabel(r'$l$', 'см')
           .ylabel(r'$\ln\left[(N-N_0)/\mathrm{мин}^{-1}\right]$')
           .title('Ослабление $\\gamma$-излучения: алюминий')
           .fit(1)
           .errors(dy=sigma_aluminium_ln, dx=dl))

# сталь: аппроксимируем только по «хорошим» точкам
plot_fe = (GostPlot(steel_l[steel_ok], steel_ln[steel_ok])
           .xlabel(r'$l$', 'см')
           .ylabel(r'$\ln\left[(N-N_0)/\mathrm{мин}^{-1}\right]$')
           .title('Ослабление $\\gamma$-излучения: сталь')
           .fit(1)
           .errors(dy=sigma_steel_ln[steel_ok], dx=dl))

# .save2pdf('fig_5_1_pb.pdf') — раскомментировать, когда графики пойдут в отчёт
plot_pb.show()
plot_al.show()
plot_fe.show()


In [ ]:
def mu_from_fit(l, y, sy, label, element):
    """МНК-прямая y = -mu*l + b; те же числа, что в легенде графика.

    sigma(mu) берётся по разбросу остатков (cov=True), а не из статистики
    счёта: chi2/ndf >> 1, т.е. систематика (билдап, толщины) доминирует.
    """
    p, cov = np.polyfit(l, y, 1, cov=True)
    chi2 = np.sum(((y - np.polyval(p, l)) / sy) ** 2)
    ndf  = len(l) - 2
    mu, s_mu = -p[0], np.sqrt(cov[0, 0])

    rho  = density[element]
    mr, s_mr = mu / rho, s_mu / rho
    tab = mu_rho_tab[element]
    print(f'{label} ({element}, rho = {rho} г/см^3)')
    print(f'  mu       = {mu:.4f} +- {s_mu:.4f} см^-1')
    print(f'  chi2/ndf = {chi2/ndf:.0f}  (>> 1 => погрешность не статистическая)')
    print(f'  mu/rho   = {mr:.4f} +- {s_mr:.4f} см^2/г   '
          f'(таблица {tab:.4f}, отклонение {100*(mr/tab-1):+.1f} %)')
    d = np.log(2) / mu
    print(f'  d_1/2    = {d:.3f} +- {d * s_mu / mu:.3f} см\n')
    return mu, s_mu


mu_pb, s_pb = mu_from_fit(lead_l, lead_ln, sigma_lead_ln, 'Свинец', 'Pb')
mu_al, s_al = mu_from_fit(aluminium_l, aluminium_ln, sigma_aluminium_ln, 'Алюминий', 'Al')
mu_fe, s_fe = mu_from_fit(steel_l[steel_ok], steel_ln[steel_ok],
                          sigma_steel_ln[steel_ok], 'Сталь', 'Fe')

# Локальный (по соседним точкам) коэффициент — видно, как он «плывёт» с толщиной:
# признак накопления рассеянного излучения (узкий пучок не идеален).
for nm, l, y, ok in [('Pb', lead_l, lead_ln, None),
                     ('Al', aluminium_l, aluminium_ln, None),
                     ('Fe', steel_l, steel_ln, steel_ok)]:
    l_, y_ = (l, y) if ok is None else (l[ok], y[ok])
    print(nm, 'mu по парам соседних точек:', np.round(-np.diff(y_) / np.diff(l_), 3))


In [ ]:
# ---------------------------------------------------------------------------
# Бюджет погрешностей mu/rho.
# Статистика счёта (~0.3 %) — НЕ главный член. Ниже — то, что в неё не входит.
# ---------------------------------------------------------------------------

delta_plate = 0.005         # штангенциркуль: половина цены деления (0.1 мм), см
plate = {'Pb': 0.5, 'Al': 2.0, 'Fe': 1.0}          # номинал пластины, см
sigma_rho = {'Pb': 0.02, 'Al': 0.01, 'Fe': 0.10}   # сталь — не чистое Fe

def budget(l, y, sy, element, mu, s_mu_fit):
    rho = density[element]
    mloc = -np.diff(y) / np.diff(l)                # локальный mu по парам точек

    parts = {
        'разброс точек вокруг прямой': s_mu_fit / mu,
        'масштаб толщины (коррелир.)': delta_plate / plate[element],
        'плотность материала':         sigma_rho[element] / rho,
        'билдап (дрейф локального mu)': mloc.std(ddof=1) / mu,
    }
    tot = np.sqrt(sum(v ** 2 for v in parts.values()))
    mr, s_mr = mu / rho, mu / rho * tot
    tabv = mu_rho_tab[element]

    print(f'{element}:')
    for k, v in parts.items():
        print(f'    {k:<30} {100*v:5.1f} %')
    print(f'    {"ИТОГО":<30} {100*tot:5.1f} %')
    print(f'  mu/rho = {mr:.4f} +- {s_mr:.4f} см^2/г  (табл. {tabv:.4f}) '
          f'-> {abs(mr - tabv) / s_mr:.1f} sigma')
    print(f'  было (только счёт):      +- {s_mu_fit/rho:.4f} '
          f'-> {abs(mr - tabv) / (s_mu_fit/rho):.1f} sigma\n')
    return s_mr

budget(lead_l, lead_ln, sigma_lead_ln, 'Pb', mu_pb, s_pb)
budget(aluminium_l, aluminium_ln, sigma_aluminium_ln, 'Al', mu_al, s_al)
budget(steel_l[steel_ok], steel_ln[steel_ok], sigma_steel_ln[steel_ok], 'Fe', mu_fe, s_fe)

# Проверка: погрешность толщины через «эффективную дисперсию».
# polyfit НЕ использует dx вообще, хотя вклад mu*sigma_l в разы больше sigma_y.
for nm, l, y, sy, mu in [('Pb', lead_l, lead_ln, sigma_lead_ln, mu_pb),
                         ('Al', aluminium_l, aluminium_ln, sigma_aluminium_ln, mu_al),
                         ('Fe', steel_l[steel_ok], steel_ln[steel_ok],
                          sigma_steel_ln[steel_ok], mu_fe)]:
    print(f'{nm}: sigma_y = {sy.mean():.4f}, mu*sigma_l = {mu*dl:.4f} '
          f'({mu*dl/sy.mean():.0f}x больше)')
